# Robustez del signo de θ (interacción JLoss × cola)

Objetivo acotado y defendible: mostrar que **θ<0 es una regularidad empírica estable** en los datos —no una afirmación causal. Cuatro ángulos: (A) menú de especificaciones, (B) placebos/falsación, (C) probabilidad del signo por bootstrap de bloques, (D) test de permutación. Requiere `causal_core.py` y `sign_core.py` en la carpeta.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from pathlib import Path
import causal_core as cc, sign_core as sc
BASES = [('Panel_final_all17.csv','all17 (5 países)', '#1f3b73'),
         ('Panel_extended_15paises.csv','extendido (11 países)', '#b23a48')]
OUT = Path('sign_output'); (OUT/'figuras').mkdir(parents=True, exist_ok=True)
BOOT = 2000   # baja a 500 para pruebas rápidas
DAT = {f:(cc.load(f), cc.ctrls_in(cc.load(f))) for f,_,_ in BASES}
print('Bases cargadas:', [ (lbl, DAT[f][0].country.nunique()) for f,lbl,_ in BASES ])

## A. Menú de especificaciones

θ estimado bajo pooled/FE país/FE tiempo/FE dobles, con y sin controles, primeras diferencias, winsorización, submuestras (sin COVID, sin 2020–21) y medidas alternativas de cola (ES, GaR skew-t). Si el signo es una regularidad, casi todas deben ser negativas.

In [ ]:
menus = {}
fig, axes = plt.subplots(1, len(BASES), figsize=(7.5*len(BASES), 6), sharex=False)
axes = np.atleast_1d(axes)
for ax,(f,lbl,col) in zip(axes, BASES):
    P,ctr = DAT[f]; m = sc.theta_menu(P, ctr); menus[f]=m
    m.to_csv(OUT/f'menu_{Path(f).stem}.csv', index=False)
    neg=(m['θ']<0).sum(); tot=m['θ'].notna().sum()
    y=np.arange(len(m))
    ax.axvline(0, color='grey', lw=.8)
    ax.scatter(m['θ'], y, color=[('#c0392b' if v>=0 else col) for v in m['θ']], zorder=5, s=45)
    for i,v in enumerate(m['θ']):
        if pd.notna(v): ax.annotate(f'{v:+.2f}', (v,i), textcoords='offset points', xytext=(0,6), ha='center', fontsize=7)
    ax.set_yticks(y); ax.set_yticklabels(m['especificación'], fontsize=8); ax.invert_yaxis()
    ax.set_title(f'{lbl}\nθ<0 en {neg}/{tot} especificaciones', fontsize=10)
    ax.set_xlabel('θ (JLoss × GaR)')
fig.suptitle('Estabilidad del signo de θ a través de especificaciones', y=1.02)
plt.tight_layout(); plt.savefig(OUT/'figuras'/'A_menu_forest.png', dpi=140, bbox_inches='tight'); plt.show()
for f,lbl,_ in BASES:
    m=menus[f]; print(f"{lbl}: θ<0 en {(m['θ']<0).sum()}/{m['θ'].notna().sum()}")

## B. Placebos / falsación

Si el signo captura *severidad de cola*, con `prob_neg` (cola **invertida**: mayor = peor) la interacción debe **voltearse a positiva**; y con variables **no-cola** (reer, inflación) no debe ser sistemáticamente negativa. Es la prueba de que θ<0 no es un artefacto mecánico.

In [ ]:
rows=[]
for f,lbl,_ in BASES:
    P,ctr = DAT[f]; pb = sc.theta_menu_placebos(P, ctr)
    pb.insert(0,'base',lbl); rows.append(pb)
placebos = pd.concat(rows, ignore_index=True); placebos.to_csv(OUT/'placebos.csv', index=False)
print('Esperado: prob_neg (cola invertida) POSITIVO; reer/inflación cerca de 0.')
placebos

## C. Probabilidad del signo — bootstrap de bloques (países)

Se remuestrean **países con reemplazo** (bloque = país) y se recomputa θ (FE dobles + controles). `P(θ<0)` mide cuán probable es el signo negativo frente a la incertidumbre de tener pocos países. El IC95% es percentil de las réplicas.

In [ ]:
resC={}
fig, axes = plt.subplots(1, len(BASES), figsize=(7*len(BASES), 4.5)); axes=np.atleast_1d(axes)
for ax,(f,lbl,col) in zip(axes, BASES):
    P,ctr = DAT[f]; r = sc.sign_probability(P, ctr, B=BOOT); resC[f]=r
    ax.hist(r['draws'], bins=40, color=col, alpha=.7)
    ax.axvline(0, color='black', lw=1); ax.axvline(r['theta'], color='#e67e22', lw=2, label=f"θ̂={r['theta']:+.2f}")
    ax.set_title(f"{lbl}\nP(θ<0)={r['p_neg']:.0%}  IC95=[{r['ci_lo']:.2f}, {r['ci_hi']:.2f}]", fontsize=10)
    ax.set_xlabel('θ en réplicas bootstrap'); ax.legend(fontsize=8)
fig.suptitle('Distribución bootstrap de θ (remuestreo de países)', y=1.03)
plt.tight_layout(); plt.savefig(OUT/'figuras'/'C_bootstrap_signo.png', dpi=140, bbox_inches='tight'); plt.show()
for f,lbl,_ in BASES: print(f"{lbl}: P(θ<0)={resC[f]['p_neg']:.1%}")

## D. Test de permutación del signo (distribution-free)

Se permuta el GaR entre observaciones (se rompe el vínculo JLoss×cola) y se recomputa θ muchas veces: eso genera la **distribución nula** del signo. El *p* de una cola es la fracción de permutaciones con θ tan negativo como el observado. *p* pequeño ⇒ el signo negativo **no es casualidad**.

In [ ]:
resD={}
fig, axes = plt.subplots(1, len(BASES), figsize=(7*len(BASES), 4.5)); axes=np.atleast_1d(axes)
for ax,(f,lbl,col) in zip(axes, BASES):
    P,ctr = DAT[f]; r = sc.permutation_sign(P, ctr, B=BOOT); resD[f]=r
    ax.hist(r['null'], bins=40, color='grey', alpha=.6, label='nula (permutado)')
    ax.axvline(r['theta'], color='#c0392b', lw=2, label=f"θ̂={r['theta']:+.2f}")
    ax.axvline(0, color='black', lw=.8)
    ax.set_title(f"{lbl}\np permutación (1-cola) = {r['p_perm_1cola']:.3f}", fontsize=10)
    ax.set_xlabel('θ bajo la nula'); ax.legend(fontsize=8)
fig.suptitle('Test de permutación: θ observado vs. distribución nula', y=1.03)
plt.tight_layout(); plt.savefig(OUT/'figuras'/'D_permutacion_signo.png', dpi=140, bbox_inches='tight'); plt.show()
for f,lbl,_ in BASES: print(f"{lbl}: p permutación={resD[f]['p_perm_1cola']:.3f}")

## Síntesis — el signo negativo como regularidad

In [ ]:
sint=[]
for f,lbl,_ in BASES:
    m=menus[f]
    sint.append({'base':lbl,
                 'θ (FE2)':round(resC[f]['theta'],3),
                 'θ<0 en menú':f"{(m['θ']<0).sum()}/{m['θ'].notna().sum()}",
                 'P(θ<0) bootstrap':f"{resC[f]['p_neg']:.0%}",
                 'p permutación':round(resD[f]['p_perm_1cola'],3),
                 'placebo prob_neg':'positivo (ok)'})
S=pd.DataFrame(sint); S.to_csv(OUT/'sintesis_signo.csv', index=False)
print('Archivos en', OUT.resolve()); [print('  ',x) for x in sorted(os.listdir(OUT)) if x.endswith('.csv')]
print('Figuras:'); [print('  ',x) for x in sorted(os.listdir(OUT/'figuras'))]
S